# Evaluation — Retrieval Recall@k

This notebook measures how well the retrieval pipeline surfaces the right
documents before the LLM ever sees them. A generation model can only answer
correctly if the relevant chunk was retrieved — retrieval quality is the
ceiling on end-to-end accuracy.

**Metric: Recall@k** — for each question, did at least one expected source
file appear in the top-k retrieved chunks? Score ranges from 0 (never finds
the right doc) to 1.0 (always finds it).

When the corpus contains duplicate valid evidence (for example, a 10-K plus a
companion annual report PDF or earnings exhibit), `expected_sources` includes
all acceptable source files so recall is not penalized for retrieving either.

The eval set is intentionally varied:

| Question type | Examples |
|---|---|
| Simple revenue lookup | Total net sales, annual revenue |
| Segment breakdown | AWS, Data Center, Google Cloud |
| Profitability | Net income, operating income |
| YoY growth | % change, direction of change |
| Operational / headcount | Employee count |
| Out-of-scope | Real-time data — model should refuse |

## 1. Setup

In [24]:
import sys
import pickle
from pathlib import Path

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

sys.path.insert(0, str(Path("..").resolve()))
from utils import Chunk, get_chroma_client

TOP_K_DENSE  = 50
TOP_K_BM25   = 50
TOP_RERANK   = 20
TOP_K_FINAL  = 5
RRF_K        = 60
CHROMA_DIR   = "../chroma_db"
COLLECTION   = "finance_rag"

cache_path = Path("../chunks_cache.pkl")
with open(cache_path, "rb") as f:
    all_chunks = pickle.load(f)

client_chroma = get_chroma_client(CHROMA_DIR)
collection    = client_chroma.get_collection(COLLECTION)

bi_encoder    = SentenceTransformer("all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

tokenized_corpus = [c.text.lower().split() for c in all_chunks]
bm25 = BM25Okapi(tokenized_corpus)

print(f"Loaded {len(all_chunks)} chunks, collection has {collection.count()} docs")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10459.11it/s]


Loaded 21386 chunks, collection has 21386 docs


## 2. Retrieval Pipeline

The same four-stage pipeline from `03_retrieval.ipynb`, copied inline so
this notebook is self-contained.

In [25]:
def dense_retrieve(query: str, k: int = TOP_K_DENSE) -> list[tuple[int, float]]:
    query_vec = bi_encoder.encode(query, convert_to_numpy=True).tolist()
    results   = collection.query(query_embeddings=[query_vec], n_results=k, include=["distances"])
    id_to_idx = {c.id: i for i, c in enumerate(all_chunks)}
    return [
        (id_to_idx[cid], 1 / (1 + dist))
        for cid, dist in zip(results["ids"][0], results["distances"][0])
        if cid in id_to_idx
    ]


def bm25_retrieve(query: str, k: int = TOP_K_BM25) -> list[tuple[int, float]]:
    scores = bm25.get_scores(query.lower().split())
    top    = scores.argsort()[::-1][:k]
    return [(int(i), float(scores[i])) for i in top]


def reciprocal_rank_fusion(
    *ranked_lists: list[tuple[int, float]],
    k: int = RRF_K,
    top_n: int = TOP_RERANK,
) -> list[int]:
    fused: dict[int, float] = {}
    for ranked in ranked_lists:
        for rank, (idx, _) in enumerate(ranked):
            fused[idx] = fused.get(idx, 0.0) + 1.0 / (k + rank + 1)
    return sorted(fused, key=fused.__getitem__, reverse=True)[:top_n]


def rerank(query: str, candidate_ids: list[int], top_n: int = TOP_K_FINAL) -> list[tuple[int, float]]:
    pairs  = [(query, all_chunks[i].text) for i in candidate_ids]
    scores = cross_encoder.predict(pairs, show_progress_bar=False)
    return sorted(zip(candidate_ids, scores.tolist()), key=lambda x: x[1], reverse=True)[:top_n]


def retrieve(query: str, top_k: int = TOP_K_FINAL) -> list[Chunk]:
    d = dense_retrieve(query)
    s = bm25_retrieve(query)
    f = reciprocal_rank_fusion(d, s)
    r = rerank(query, f, top_n=top_k)
    return [all_chunks[idx] for idx, _ in r]


print("retrieve() ready")

retrieve() ready


## 3. Eval Set

20 questions verified against the raw filings. Answers span 8 companies
and 5 question categories. The final entry is deliberately out-of-scope
(real-time data) and should produce a retrieval near-miss plus a model
refusal downstream.

In [26]:
capstone_eval = [
    {
        "q": "What were Amazon's total net sales for fiscal year 2025?",
        "expected_sources": ["AMZN_10-K_2026-02-06.htm", "Amazon-2025-Annual-Report.pdf"],
        "note": "$716,924M total net sales",
    },
    {
        "q": "How much revenue did NVIDIA generate in fiscal year 2026?",
        "expected_sources": ["NVDA_10-K_2026-02-25.htm"],
        "note": "$215.9B, up 65% YoY",
    },
    {
        "q": "What was Meta's total revenue for the year ended December 31, 2025?",
        "expected_sources": ["META_10-K_2026-01-29.htm", "Meta-12-31-2025-Exhibit-99-1-FINAL.pdf"],
        "note": "$200,966M total revenue",
    },
    {
        "q": "What were Alphabet's consolidated total revenues for the calendar year ended December 31, 2025?",
        "expected_sources": ["GOOGL_10-K_2026-02-05.htm"],
        "note": "$402,836M",
    },
    {
        "q": "What was Microsoft's total revenue in fiscal year 2025?",
        "expected_sources": ["MSFT_10-K_2025-07-30.htm"],
        "note": "$281,724M (FY ends June 2025)",
    },
    {
        "q": "What was AMD's total net revenue for fiscal year 2025?",
        "expected_sources": ["AMD_10-K_2026-02-04.htm", "AMD Q4'25 Earnings Slides FINAL.pdf"],
        "note": "$34,639M total net revenue",
    },
    {
        "q": "How much did AWS contribute to Amazon's net sales in fiscal year 2025?",
        "expected_sources": ["AMZN_10-K_2026-02-06.htm", "Amazon-2025-Annual-Report.pdf"],
        "note": "AWS net sales $128,725M",
    },
    {
        "q": "What was NVIDIA's Data Center revenue for fiscal year 2026?",
        "expected_sources": ["NVDA_10-K_2026-02-25.htm"],
        "note": "$193,737M",
    },
    {
        "q": "What was Google Cloud revenue in Alphabet's fiscal year 2025?",
        "expected_sources": ["GOOGL_10-K_2026-02-05.htm"],
        "note": "$58,705M",
    },
    {
        "q": "What were Apple's Services net sales in Q1 fiscal year 2026?",
        "expected_sources": ["AAPL_10-Q_2026-01-30.htm"],
        "note": "$30,013M (quarter ended December 27, 2025)",
    },
    {
        "q": "What was AMD's Data Center segment revenue in fiscal year 2025?",
        "expected_sources": ["AMD_10-K_2026-02-04.htm", "AMD Q4'25 Earnings Slides FINAL.pdf"],
        "note": "$16,635M Data Center net revenue",
    },
    {
        "q": "What was NVIDIA's full-year net income for the fiscal year ended January 2026?",
        "expected_sources": ["NVDA_10-K_2026-02-25.htm"],
        "note": "$120,067M",
    },
    {
        "q": "What was Alphabet's consolidated operating income for the year ended December 31, 2025?",
        "expected_sources": ["GOOGL_10-K_2026-02-05.htm"],
        "note": "$129,039M, operating margin 32%",
    },
    {
        "q": "What was Tesla's annual net income for the fiscal year ended December 31, 2025?",
        "expected_sources": ["TSLA_10-K_2026-01-29.htm"],
        "note": "$3,855M (down from $7,153M in FY2024)",
    },
    {
        "q": "By what percentage did NVIDIA's revenue grow in fiscal year 2026?",
        "expected_sources": ["NVDA_10-K_2026-02-25.htm"],
        "note": "65% growth",
    },
    {
        "q": "Did Tesla's total revenues increase or decrease from fiscal year 2024 to 2025?",
        "expected_sources": ["TSLA_10-K_2026-01-29.htm"],
        "note": "Decreased \u2014 $97,690M (2024) to $94,827M (2025), down ~3%",
    },
    {
        "q": "By how much did AMD's net revenue grow from fiscal year 2024 to 2025?",
        "expected_sources": ["AMD_10-K_2026-02-04.htm", "AMD Q4'25 Earnings Slides FINAL.pdf"],
        "note": "From $25,785M to $34,639M \u2014 up 34%",
    },
    {
        "q": "How many employees did NVIDIA have as of fiscal year 2026?",
        "expected_sources": ["NVDA_10-K_2026-02-25.htm"],
        "note": "~42,000 employees in 38 countries",
    },
    {
        "q": "How many full-time employees did Alphabet have as of December 31, 2025?",
        "expected_sources": ["GOOGL_10-K_2026-02-05.htm"],
        "note": "190,820 employees",
    },
    {
        "q": "What is Tesla's current share price?",
        "expected_sources": [],
        "note": "Out-of-scope \u2014 real-time price not in filings; model must refuse",
    },
]

print(f"Eval set: {len(capstone_eval)} questions")
for i, item in enumerate(capstone_eval, 1):
    cat = "OOS " if not item["expected_sources"] else item["expected_sources"][0][:28]
    print(f"  {i:2}. [{cat:<28}] {item['q'][:55]}")


Eval set: 20 questions
   1. [AMZN_10-K_2026-02-06.htm    ] What were Amazon's total net sales for fiscal year 2025
   2. [NVDA_10-K_2026-02-25.htm    ] How much revenue did NVIDIA generate in fiscal year 202
   3. [META_10-K_2026-01-29.htm    ] What was Meta's total revenue for the year ended Decemb
   4. [GOOGL_10-K_2026-02-05.htm   ] What were Alphabet's consolidated total revenues for th
   5. [MSFT_10-K_2025-07-30.htm    ] What was Microsoft's total revenue in fiscal year 2025?
   6. [AMD_10-K_2026-02-04.htm     ] What was AMD's total net revenue for fiscal year 2025?
   7. [AMZN_10-K_2026-02-06.htm    ] How much did AWS contribute to Amazon's net sales in fi
   8. [NVDA_10-K_2026-02-25.htm    ] What was NVIDIA's Data Center revenue for fiscal year 2
   9. [GOOGL_10-K_2026-02-05.htm   ] What was Google Cloud revenue in Alphabet's fiscal year
  10. [AAPL_10-Q_2026-01-30.htm    ] What were Apple's Services net sales in Q1 fiscal year 
  11. [AMD_10-K_2026-02-04.htm     ] What was AM

## 4. Run Evaluation

`eval_recall` iterates over the eval set and prints a HIT / MISS line for
each question. Out-of-scope entries are shown separately and excluded from
the recall denominator — they test the model's refusal behaviour, not
retrieval coverage.

In [27]:
def eval_recall(eval_set: list, k: int = 5) -> float:
    # hits counts how many expected source-documents were actually retrieved.
    # scoreable counts how many expected source-documents we asked retrieval to find.
    # This means multi-source questions get full credit only if all expected
    # sources appear in the retrieved set, but they can still earn partial credit.
    hits = 0
    scoreable = 0

    for item in eval_set:
        retrieved = retrieve(item["q"], top_k=k)
        retrieved_sources = {c.source for c in retrieved}

        # Out-of-scope questions are excluded from recall because they test
        # refusal behavior rather than source retrieval.
        if not item["expected_sources"]:
            print(f"[OOS ] {item['q']}")
            print(f"       retrieved: {sorted(retrieved_sources)[:3]}")
            print()
            continue

        expected = set(item["expected_sources"])
        matched = expected & retrieved_sources

        # Each expected source contributes one unit to the denominator.
        # A question with 2 valid source files is worth 2 possible hits.
        scoreable += len(expected)

        # Add one hit for each expected source we actually retrieved.
        hits += len(matched)

        # For the printed label, require every expected source to be present.
        full_hit = matched == expected
        label = "HIT " if full_hit else "MISS"
        print(f"[{label}] {item['q'][:65]}")
        if not full_hit:
            print(f"       expected : {sorted(expected)}")
            print(f"       matched  : {sorted(matched)}")
            print(f"       retrieved: {sorted(retrieved_sources)}")

    recall = hits / scoreable if scoreable else 0.0
    print(f"\n{'─'*70}")
    print(f"Recall@{k}: {hits}/{scoreable} = {recall:.0%}")
    return recall


recall = eval_recall(capstone_eval, k=5)

[HIT ] What were Amazon's total net sales for fiscal year 2025?
[HIT ] How much revenue did NVIDIA generate in fiscal year 2026?
[HIT ] What was Meta's total revenue for the year ended December 31, 202
[MISS] What were Alphabet's consolidated total revenues for the calendar
       expected : ['GOOGL_10-K_2026-02-05.htm']
       matched  : []
       retrieved: ['GOOGL_10-K_2025-02-05.htm', 'goog-10-q-q1-2025.pdf']
[HIT ] What was Microsoft's total revenue in fiscal year 2025?
[MISS] What was AMD's total net revenue for fiscal year 2025?
       expected : ["AMD Q4'25 Earnings Slides FINAL.pdf", 'AMD_10-K_2026-02-04.htm']
       matched  : ['AMD_10-K_2026-02-04.htm']
       retrieved: ['AMD_10-K_2025-02-05.htm', 'AMD_10-K_2026-02-04.htm', 'AMD_10-Q_2024-10-30.htm', 'AMD_10-Q_2025-05-07.htm', 'AMD_10-Q_2025-11-05.htm']
[HIT ] How much did AWS contribute to Amazon's net sales in fiscal year 
[HIT ] What was NVIDIA's Data Center revenue for fiscal year 2026?
[HIT ] What was Google Cloud reve

## 5. Results Analysis & Improvement Recommendations

### Scores

| Metric | Score | Notes |
|---|---|---|
| Question-level Recall@5 | **74%** (14/19) | Full credit only if ALL expected sources retrieved |
| Strict Source-level Recall@5 | **80%** (20/25) | Partial credit per source file |

89% question-level was achievable with single-source questions only; adding multi-source
expectations (earnings slide PDFs) exposed two distinct failure modes.

---

### Failure Mode 1 — Year Disambiguation (GOOGL, 2 misses)

Both Alphabet misses retrieve `GOOGL_10-K_2025-02-05.htm` (the FY **2024** annual report,
confusingly dated Feb 2025) instead of `GOOGL_10-K_2026-02-05.htm` (FY 2025).

The root cause: `total revenues` and `operating income` appear in near-identical prose in
consecutive annual reports. Neither the bi-encoder nor BM25 can distinguish them — the
semantic content is the same, only the numbers differ.

Note that `Google Cloud revenue` **did** hit the correct filing: segment names provide enough
distinctive signal. Generic financial terms do not.

**Suggested fixes**

| Fix | Complexity | Expected gain |
|---|---|---|
| Store `filing_date` as chunk metadata; post-filter by date range at query time | Low | High — rules out wrong-year filings entirely |
| Prepend `[Source: GOOGL_10-K_2026-02-05.htm, filed 2026-02-05]` to each chunk during indexing | Low | Medium — adds date signal to BM25 and dense vectors |
| Query rewriting: extract the target date and append `filed:{year}` as a filter | Medium | High |

---

### Failure Mode 2 — Earnings Slide PDFs Not Retrieved (AMD, 3 misses)

All three AMD misses find the 10-K but never surface `AMD Q4'25 Earnings Slides FINAL.pdf`.
The 10-K is always in the retrieved set; the PDF is consistently absent.

Likely causes:
- **Sparse slide text**: earnings slides are bullet-heavy and figure-heavy; the prose per
  chunk is thin, producing weak embeddings relative to the dense 10-K narrative.
- **Duplicate signal**: the 10-K contains the same figures with more surrounding context,
  so it always scores higher and the slides never compete for a top-5 slot.

**Suggested fixes**

| Fix | Complexity | Expected gain |
|---|---|---|
| Use smaller `CHUNK_SIZE` for PDFs identified as slide decks (e.g. 300 tokens vs 800) | Low | Medium — preserves slide-level granularity |
| Assign document-type metadata (`10-K`, `earnings-slides`) and add a diversity rule: require at least one chunk from each unique document type | Medium | High — forces slides into the result set |
| MMR (Maximal Marginal Relevance) reranking: penalise redundant chunks from the same source, favouring coverage over similarity | Medium | Medium — naturally surfaces the second source |

---

### Other Observations

- **OOS behaviour is correct**: the out-of-scope question (`Tesla's current share price`)
  retrieved a TSLA 10-K, which is expected — real-time data simply isn't in the corpus.
  Downstream, the generation guardrail should still refuse, since no chunk contains a
  live price.

- **k=5 is tight for multi-source questions**: questions that require *two* source files
  to both appear in the top-5 are structurally harder. Raising to `k=8` or `k=10` would
  likely recover the AMD slide misses with no other changes.

---

### Prioritised Action List

1. **Raise `k` to 8 or 10** — zero code changes, likely recovers AMD slide misses
2. **Prepend filing metadata to chunks at index time** — fixes year disambiguation permanently
3. **Document-type diversity rule in reranker** — ensures slide decks aren't crowded out by 10-Ks

                                                                                                                                                                                   
  ### What we changed, and why:                                                                                                                                                                  
                                                            
  **1. Added ticker pre-filtering (new step before everything)**                                                                                                                               
  
  The notebook retrieves from the entire 21k-chunk corpus. For a question about Meta, BM25 and dense both match on the generic tokens "revenue" and "December 31, 2025" — so TSLA and AMZN   
  chunks beat Meta's own chunks. We added:                  
  ticker = _detect_ticker(query)   # "meta's" → "META"                                                                                                                                       
  valid_idxs = self._ticker_idxs["META"]   # set of only Meta chunk indices
  sources    = self._ticker_sources["META"] # list of Meta filenames for ChromaDB                                                                                                            
  Now both retrievers only search within that company's documents.                                                                                                                           
                                                                                                                                                                                             
  **2. Larger candidate pools (50 → 100 for both dense and BM25)**                                                                                                                               
                                                                                                                                                                                             
  Even after filtering to Meta-only (~790 chunks), the answer chunk from Exhibit-99-1-FINAL.pdf ranked below position 50 in dense search. Doubling the pool ensures more candidates enter the
   RRF step.                                                                                                                                                                                 
                                                                                                                                                                                             
  **3. Bigger RRF → reranker window (20 → 50)**                                                                                                                                                  
  
  The $200,966 table chunk had RRF rank 21 in the original setup — just outside the 20 passed to the cross-encoder. Expanding to 50 candidates means the reranker actually sees it.          
                                                            
  **4. More final results to the LLM (5 → 15)**                                                                                                                                                  
                                                            
  Even after reranking, the table chunk scores lower than prose chunks because ms-marco-MiniLM-L-6-v2 was trained on natural language, not markdown tables with | separators. Returning 15   
  instead of 5 means moderately-scored-but-directly-relevant chunks still make it into the LLM's context.
                                                                                                                                                                                             
  ---                                                       
  The notebook's pipeline diagram should now read:
  Query                                                                                                                                                                                      
    ├─► Ticker detection  → pre-filter to one company's chunks
    ├─► Dense  → top-100 from that company                                                                                                                                                   
    ├─► BM25   → top-100 from that company                  
    ├─► RRF    → top-50 merged            
    └─► Rerank → top-15 final 